# Validating the exoplanet simulator physics

This notebook is the validation layer for the exoplanet tools. It runs in
JupyterLite, so everything happens in the browser with no server.

**What this notebook is for.** The simulator's numbers appear in front of
students as if they were facts. This is where you check that they deserve
to be. Three things happen below:

1. The Python implementation is run against `test_cases.json`, the same
   contract the JavaScript test suite uses. Agreement means the two
   implementations have not drifted apart.
2. Model output is compared against published results. This is the part
   that checks *correctness* rather than consistency, and it is the part
   worth growing.
3. A few diagnostics that are awkward in the browser: bifurcation
   diagrams, parameter sweeps, hysteresis.

**A caveat to keep in view.** `physics.py` and `physics.js` were written
from the same understanding. If that understanding is wrong, both will be
wrong together and every consistency check will still pass. Only the
cases marked `"source": "published"` can catch that, because only those
compare against something outside this project.

**Files this notebook needs**, in the same directory:
`physics.py`, `test_cases.json`.

### Before anything else: find the supporting files

This notebook needs `physics.py` and `test_cases.json`. Normally they sit
beside it and the import just works. JupyterLite runs Python inside the
browser with its own virtual filesystem, so occasionally it does not, and a
plain `ImportError` on the first cell is an unhelpful way to find that out.

The cell below tries the local files first and falls back to fetching them
from GitHub. Set `REPO` to your own repository if you fork or rename it.

In [ ]:
import sys, os, json

REPO   = "BuchananGCSC/jupyterlite"      # where these files actually live
BRANCH = "main"
RAW    = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}/content/"

def _fetch(name):
    """Download a support file into the working directory."""
    try:
        from pyodide.http import open_url          # JupyterLite / Pyodide
        text = open_url(RAW + name).read()
    except ImportError:
        from urllib.request import urlopen         # ordinary Python
        text = urlopen(RAW + name).read().decode("utf8")
    with open(name, "w") as handle:
        handle.write(text)
    return text

for name in ("physics.py", "test_cases.json"):
    if os.path.exists(name):
        print(f"{name}: found locally")
    else:
        try:
            _fetch(name)
            print(f"{name}: fetched from {REPO}")
        except Exception as exc:
            print(f"{name}: NOT AVAILABLE -- {exc}")
            print("  Upload it into this JupyterLite session using the file "
                  "browser on the left, then re-run this cell.")

if "." not in sys.path:
    sys.path.insert(0, ".")

## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import physics as P

plt.rcParams.update({'figure.figsize': (9, 5), 'axes.grid': True, 'grid.alpha': 0.3})
contract = P.load_contract('test_cases.json')
print(f"contract loaded: {len(contract['cases'])} cases")

## 2. Run the shared contract

In [ ]:
results, n_pass, n_fail, n_skip = P.run_contract(contract['cases'])

print(f"pass {n_pass}   fail {n_fail}   skipped {n_skip}\n")
for case_id, status, detail in results:
    if status != 'PASS':
        print(f"{status:6} {case_id:45} {detail}")

if n_fail == 0:
    print("Every implemented case passes.")

If something fails here but passes in the JavaScript suite, the two
implementations have drifted and one of them changed without the other.
If it fails in both, the contract is telling you the physics is wrong,
which is the more interesting case.

## 3. Habitable zone against Kopparapu et al. 2013

The published figure to check against is the habitable zone boundaries as
a function of stellar effective temperature. The four curves below should
reproduce the familiar wedge shape: boundaries move outward in
flux-normalised terms for cooler stars because water ice and snow are
less reflective in the near-infrared, where an M dwarf puts most of its
light.

The horizontal marks are the solar values the test suite pins.

In [ ]:
teffs = np.linspace(2600, 7200, 300)
limits = ['recentVenus', 'runawayGreenhouse', 'maxGreenhouse', 'earlyMars']
labels = ['Recent Venus (optimistic inner)', 'Runaway greenhouse (conservative inner)',
          'Maximum greenhouse (conservative outer)', 'Early Mars (optimistic outer)']

fig, ax = plt.subplots()
for limit, label in zip(limits, labels):
    ax.plot(teffs, [P.seff_at(limit, t) for t in teffs], label=label)
ax.axvline(5772, color='k', lw=0.8, ls='--')
ax.text(5800, 1.6, 'Sun', fontsize=9)
ax.set_xlabel('Stellar effective temperature (K)')
ax.set_ylabel('Effective stellar flux $S_{eff}$ (Earth = 1)')
ax.set_title('Habitable zone limits, Kopparapu et al. 2013 parameterisation')
ax.legend(fontsize=8)
plt.show()

for name in limits:
    print(f"{name:20} S_eff at the Sun = {P.seff_at(name, 5780):.4f}  "
          f"-> {np.sqrt(1/P.seff_at(name, 5780)):.3f} AU")

Sanity check the whole zone for a few real stars. Compare these against
the literature values for each system before trusting them in class.

In [ ]:
systems = [('Sun', 1.0, {'Venus': 0.72, 'Earth': 1.00, 'Mars': 1.52}),
           ('Proxima Centauri', 0.122, {'Proxima b': 0.0485}),
           ('TRAPPIST-1', 0.089, {'TRAPPIST-1e': 0.029, 'TRAPPIST-1f': 0.038}),
           ('a K dwarf', 0.7, {})]

for name, mass, planets in systems:
    hz = P.habitable_zone(mass)
    print(f"{name} ({mass} Msun, Teff {hz['teff']:.0f} K)")
    print(f"   conservative {hz['conservative'][0]:.4f} - {hz['conservative'][1]:.4f} AU")
    print(f"   optimistic   {hz['optimistic'][0]:.4f} - {hz['optimistic'][1]:.4f} AU")
    for pname, a in planets.items():
        inside_c = hz['conservative'][0] <= a <= hz['conservative'][1]
        inside_o = hz['optimistic'][0] <= a <= hz['optimistic'][1]
        verdict = 'conservative HZ' if inside_c else ('optimistic HZ only' if inside_o else 'outside')
        print(f"      {pname:14} {a:7.4f} AU  {verdict}")
    print()

## 4. The snowball bifurcation

This is where the old fixed-iteration solver went wrong, so it is worth
looking at directly. Sweep the solar constant up and then back down,
starting each run from the previous state, and the two branches separate:
a partially glaciated planet and a fully frozen one can both be steady
states at the same insolation. That gap is the ice-albedo bifurcation and
it is real physics, not a numerical artefact.

It is also why a solver that stops after a fixed number of iterations is
dangerous here. Near the fold, convergence is slow, and stopping early
lands you on neither branch.

In [ ]:
def equilibrium_from(state, S0, d_rel=0.35, base_albedo=0.30):
    """One equilibrium solve, warm-started from a previous ice distribution."""
    lats = P.latitude_grid()
    q = P.daily_mean_insolation(S0, 23.44, lats)
    lo, di, up = P.diffusion_operator(lats, d_rel, P.B_OLR)
    frac = state.copy()
    for _ in range(400):
        alpha = frac * P.ALPHA_ICE + (1 - frac) * base_albedo
        temps = P.tridiag_solve(lo, di, up, q * (1 - alpha) - P.A_OLR)
        new = frac + 0.5 * (P.ice_fraction(temps) - frac)
        if np.max(np.abs(new - frac)) < 1e-6:
            frac = new
            break
        frac = new
    weights = np.cos(np.radians(lats))
    return frac, float(np.sum(temps * weights) / np.sum(weights))

fluxes_up = np.arange(900, 1500, 5.0)
state = np.ones(len(P.latitude_grid()))          # start frozen
warming = []
for S0 in fluxes_up:
    state, mean_t = equilibrium_from(state, S0)
    warming.append(mean_t)

cooling = []
for S0 in fluxes_up[::-1]:                        # come back down from warm
    state, mean_t = equilibrium_from(state, S0)
    cooling.append(mean_t)

fig, ax = plt.subplots()
ax.plot(fluxes_up, warming, label='warming from a frozen start')
ax.plot(fluxes_up, cooling[::-1], label='cooling from a warm start')
ax.axvline(P.S0_SUN, color='k', lw=0.8, ls='--')
ax.text(P.S0_SUN + 8, -40, 'present-day Earth', fontsize=9)
ax.set_xlabel('Solar constant $S_0$ (W m$^{-2}$)')
ax.set_ylabel('Global mean temperature (degC)')
ax.set_title('Ice-albedo hysteresis: two stable states at the same insolation')
ax.legend()
plt.show()

## 5. What the seasonal fix actually changed

The old rotating-planet tab solved for radiative equilibrium at a fixed
solstice declination. That answers the question "what temperature would
this latitude reach if the sun stopped where it is and stayed there," and
under polar day the answer runs away. Adding heat capacity and marching
through the year answers the question that was actually being asked.

The gap between the two curves at the summer pole is the size of the
error the old tab was showing to students.

In [ ]:
eq = P.lat_profile_equilibrium(obliquityDeg=23.44, dRel=0.20)
se = P.lat_profile_seasonal(obliquityDeg=23.44)

fig, ax = plt.subplots()
ax.plot(eq['lats'], eq['tempsCRaw'], label='fixed-declination equilibrium (the old tab)')
ax.plot(se['lats'], se['solsticeC'], label='seasonal model, at solstice')
ax.plot(se['lats'], se['annualMeanC'], label='seasonal model, annual mean', ls='--')
ax.axhline(0, color='k', lw=0.6)
ax.set_xlabel('Latitude (degrees)')
ax.set_ylabel('Temperature (degC)')
ax.set_title('Northern summer solstice, Earth-like planet')
ax.legend(fontsize=8)
plt.show()

print(f"summer pole, fixed-declination equilibrium: {eq['tempsCRaw'][-1]:6.1f} degC")
print(f"summer pole, seasonal model:                {se['solsticeC'][-1]:6.1f} degC")
print(f"observed Arctic July mean:                  ~   0    degC")
print(f"\nglobal mean, seasonal model: {se['globalMeanC']:.1f} degC   (observed 14)")
print(f"converged after {se['yearsRun']} model years: {se['converged']}")

### Seasonal cycle by latitude

The Hovmoller diagram the simulator draws, but produced independently
here. Check the phase: warmest northern latitudes should fall in June and
July, warmest southern latitudes in December and January. Getting this
backwards is an easy sign error to miss on a colour plot.

In [ ]:
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, ptype in zip(axes, ['earth', 'desert', 'ocean']):
    s = P.lat_profile_seasonal(planetType=ptype)
    steps = s['stepsPerYear']
    im = ax.pcolormesh(np.linspace(0, 12, steps), s['lats'], s['monthsRaw'].T,
                       cmap='RdBu_r', vmin=-60, vmax=40, shading='auto')
    ax.set_title(f"{P.PLANET_TYPES[ptype]['label']}  (global mean {s['globalMeanC']:.1f} degC)")
    ax.set_xticks(np.arange(12) + 0.5)
    ax.set_xticklabels(months, fontsize=7)
    ax.set_xlabel('Month')
axes[0].set_ylabel('Latitude (degrees)')
fig.colorbar(im, ax=axes, label='Temperature (degC)')
plt.show()

## 6. Obliquity

Published energy-balance and GCM work finds that high obliquity warms a
planet's poles and cools its equator while leaving the global mean nearly
unchanged, because obliquity redistributes sunlight rather than adding
any. If your curve shows the global mean collapsing at high obliquity,
the model has a runaway seasonal-ice artefact.

In [ ]:
obliquities = np.arange(0, 91, 5.0)
rows = [P.lat_profile_seasonal(obliquityDeg=float(o)) for o in obliquities]

fig, ax = plt.subplots()
ax.plot(obliquities, [r['annualMeanC'][-1] for r in rows], label='north pole, annual mean')
ax.plot(obliquities, [r['annualMeanC'][len(r['lats'])//2] for r in rows], label='equator, annual mean')
ax.plot(obliquities, [r['globalMeanC'] for r in rows], label='global mean', ls='--', color='k')
ax.axvline(23.44, color='grey', lw=0.8)
ax.text(24.5, -20, "Earth", fontsize=9)
ax.set_xlabel('Obliquity (degrees)')
ax.set_ylabel('Temperature (degC)')
ax.set_title('Obliquity redistributes heat without changing the total')
ax.legend()
plt.show()

## 7. Tidally locked planets

Two things to check. First, energy conservation: the day-side insolation
pattern must average to $S_0/4$ over the whole sphere, or a locked planet
is quietly receiving a different amount of energy than the same planet
unlocked. Second, the day-night contrast as a function of heat transport,
which is the parameter students will actually reach for.

The dashed line marks roughly where CO2 freezes out at one bar. Below it
the atmosphere condenses onto the night side and the diffusive picture
stops applying. That is a real prediction, and one of the standard
arguments about thin atmospheres on locked worlds.

In [ ]:
print(f"area-weighted mean of Q/S0 on a locked planet: "
      f"{P.tidalLockedMeanInsolationFraction():.5f}   (must be 0.25)")

d_values = np.linspace(0.02, 0.7, 40)
S0_proxima = P.effective_s0(0.122, 0.0485)
subs, antis = [], []
for d in d_values:
    r = P.tidally_locked_profile(S0=S0_proxima, dRel=float(d))
    subs.append(r['tempsCRaw'][-1])
    antis.append(r['tempsCRaw'][0])

fig, ax = plt.subplots()
ax.plot(d_values, subs, label='substellar point')
ax.plot(d_values, antis, label='antistellar point')
ax.axhline(-140, color='r', ls=':', lw=1)
ax.text(0.4, -133, 'CO$_2$ frost point at 1 bar', color='r', fontsize=8)
ax.set_xlabel('Heat transport parameter $D_{rel}$')
ax.set_ylabel('Temperature (degC)')
ax.set_title(f'Proxima b conditions ($S_0$ = {S0_proxima:.0f} W m$^{{-2}}$)')
ax.legend()
plt.show()

## 8. Atmospheric escape: what a planet can hold

This is the piece that gives planet mass consequences. Before it, mass
reached only the tidal-locking radius and the dynamo estimate, so a small
planet was small and nothing followed from it.

Escape velocity is exact arithmetic. The retention *rule* is a rule of
thumb: a gas survives for billions of years if the escape velocity is
comfortably above the typical thermal speed of its molecules, because the
fast tail of the Maxwell-Boltzmann distribution is always leaking away.
`JEANS_RETENTION_FACTOR = 6` is a choice, and the only honest test of it
is whether it reproduces a solar system we already know the answers for.

In [ ]:
bodies = [
    #  name        M/Me     rho    S0 (W/m^2)   what it actually has
    ('Mercury',    0.055,   5.43,  9130,  'essentially none'),
    ('Venus',      0.815,   5.24,  2601,  'thick CO2'),
    ('Earth',      1.000,   5.51,  1361,  'N2 + O2, no H2 or He'),
    ('Mars',       0.107,   3.93,   586,  'thin CO2, no water'),
    ('Titan',      0.0225,  1.88,    15,  'thick N2 -- and it is COLD'),
    ('Jupiter',  317.800,   1.33,    50,  'H2 and He'),
]

print(f"{'body':9} {'v_esc':>7} {'T_exo':>7}   keeps")
print('-' * 74)
for name, m, rho, s0, actual in bodies:
    r = P.gas_retention(m, rho, s0)
    keeps = ' '.join(g['label'] for g in r['species'].values() if g['retained']) or 'nothing'
    print(f"{name:9} {r['escapeVelocityMS']/1000:6.1f}k {r['exosphereK']:6.0f}K   {keeps}")
    print(f"{'':9} {'':7} {'':7}   actually: {actual}")

print()
print("Earth is the case to check: it should keep N2, O2, CO2 and water,")
print("lose hydrogen outright, and lose helium only marginally -- which is")
print("exactly what Earth does. Titan is the useful counter-example: far too")
print("small to hold nitrogen at Earth's temperature, and it holds a thick")
print("nitrogen atmosphere anyway, because it is cold. Retention is not about")
print("gravity alone, and the model gets that right for the right reason.")

### The classic diagram

Escape velocity against exosphere temperature, with one line per gas
showing the escape velocity needed to keep it. A body above a gas's line
keeps that gas; below it, the gas leaks away. This is the figure the
retention rule *is*, and it is worth putting in front of students
directly — the diagonal lines are the whole argument.

In [ ]:
temps = np.linspace(50, 2000, 400)
fig, ax = plt.subplots(figsize=(9.5, 6))

for key in ['H2', 'He', 'CH4', 'H2O', 'N2', 'CO2']:
    mu = P.GAS_SPECIES[key]['mu']
    needed = [P.JEANS_RETENTION_FACTOR * P.thermal_speed(mu, t) / 1000 for t in temps]
    ax.plot(temps, needed, lw=1.6, label=P.GAS_SPECIES[key]['label'])
    ax.annotate(P.GAS_SPECIES[key]['label'], (temps[-1], needed[-1]),
                textcoords='offset points', xytext=(4, -3), fontsize=9)

for name, m, rho, s0, _ in bodies:
    r = P.gas_retention(m, rho, s0)
    ax.plot(r['exosphereK'], r['escapeVelocityMS'] / 1000, 'ko', ms=6)
    ax.annotate(name, (r['exosphereK'], r['escapeVelocityMS'] / 1000),
                textcoords='offset points', xytext=(6, 5), fontsize=9)

ax.set_xlabel('exosphere temperature (K)')
ax.set_ylabel('escape velocity (km/s)')
ax.set_yscale('log')
ax.set_title('A body keeps every gas whose line lies below it')
ax.set_xlim(0, 2200)
fig.tight_layout()
plt.show()

## 9. Why there is no oxygen term in the forcing

Students ask this, and the answer is one of the most useful facts in the
whole model, so it is worth being able to demonstrate rather than assert.

Nitrogen and oxygen are **homonuclear diatomics** — two identical atoms.
The molecule is perfectly symmetric, it has no dipole moment, and its one
vibration does not change the charge distribution. Nothing about that
motion couples to infrared light, so N2 and O2 cannot absorb the heat a
planet is trying to radiate away, and no quantity of either warms it.

CO2 can bend into an asymmetric shape, methane likewise, and water is
bent to begin with. Those three absorb. That is the entire difference.

So the missing oxygen slider is not a simplification — putting one in the
forcing would be *wrong*. Oxygen does change the planet, through ozone
and ultraviolet shielding, which is where the model puts it.

In [ ]:
print('gas    mu     absorbs infrared?')
for k, g in P.GAS_SPECIES.items():
    print(f"  {g['label']:4} {g['mu']:6.2f}   {'yes' if g['ir_active'] else 'no'}")

print()
print('forcing while oxygen goes from nothing to almost everything:')
for o2 in [0, 1_000, 10_000, 209_000, 900_000]:
    mix = P.atmosphere_mixture(co2Ppm=400, o2Ppm=o2)
    f = P.greenhouse_forcing_mix(pressureBar=1.0, co2Ppm=400)
    print(f"  O2 = {o2/1e4:5.1f}%   mean mu {mix['meanMu']:5.2f}   "
          f"forcing {f['total']:6.3f} W/m2   ozone shielding: {mix['ozone']['label']}")

print()
print('and the same amounts of CO2, for contrast:')
for co2 in [400, 10_000, 209_000, 900_000]:
    f = P.greenhouse_forcing_mix(pressureBar=1.0, co2Ppm=co2)
    print(f"  CO2 = {co2/1e4:5.1f}%  forcing {f['total']:6.2f} W/m2")

print()
print('Oxygen moves the mean molecular weight, and therefore retention, and')
print('it switches on ultraviolet shielding. It does not move the forcing at')
print('all, and that is the correct answer rather than a missing feature.')

## 10. The surface axis

`PLANET_TYPES` used to bundle albedo, heat transport and heat storage
behind three names. All three follow from one physical quantity — how
much surface water there is — and the three old types turn out to be
exactly three points on that axis. Worth confirming, since the claim is
that nothing was lost when the dropdown became a slider.

In [ ]:
print('the three former types, reproduced from the water axis alone:')
for w, name in [(0.0, 'desert'), (0.5, 'earth'), (1.0, 'ocean')]:
    s, t = P.surface_properties(w), P.PLANET_TYPES[name]
    exact = (abs(s['albedo'] - t['albedo']) < 1e-12
             and abs(s['dRel'] - t['defaultD']) < 1e-12
             and abs(s['mixedLayerM'] - t['mixedLayerM']) < 1e-12)
    print(f"  w={w:.1f}  {t['label']:12} albedo {s['albedo']:.3f}  "
          f"D {s['dRel']:.3f}  mixed layer {s['mixedLayerM']:5.1f} m   "
          f"{'EXACT' if exact else '*** MISMATCH ***'}")

ws = np.linspace(0, 1, 200)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
for ax, key, label in zip(axes, ['albedo', 'dRel', 'mixedLayerM'],
                          ['albedo', 'heat transport D', 'heat storage (m)']):
    ax.plot(ws, [P.surface_properties(w)[key] for w in ws], lw=2)
    for w, name in [(0.0, 'desert'), (0.5, 'earth'), (1.0, 'ocean')]:
        ax.plot(w, P.surface_properties(w)[key], 'o', color='crimson', ms=7)
    ax.set_xlabel('surface water fraction')
    ax.set_title(label, fontsize=10)
    if key != 'albedo':
        ax.set_yscale('log')
fig.tight_layout()
plt.show()

## 11. Things still worth checking

The contract is only as good as the cases in it. Places where the model
is currently unvalidated, roughly in order of how much they matter:

- **The linear OLR fit.** `A_OLR` and `B_OLR` are tuned to Earth. How far
  from Earth-like conditions do they stay usable? A comparison against a
  line-by-line radiative transfer result, even at a handful of points,
  would set the honest edges of the `MODEL_RANGE` guard rails rather than
  the guessed ones.
- **The pressure term.** `B_PRESSURE = 4.0` W/m^2 per e-fold is a fitted
  number with no source. Check it against Venus and against Titan. Note
  that `greenhouse_forcing` still double-counts part of the CO2 term;
  `greenhouse_forcing_mix` does not, because CO2 enters there as a partial
  pressure and the broadening term is deliberately not added on top.
- **The heat transport parameter.** `D_SCALE` is tuned so Earth comes out
  right at one obliquity and one rotation rate. Real meridional transport
  depends on rotation rate, which the model does not represent at all.
- **The dynamo heuristic.** Compare against the scaling laws in McIntyre,
  Lineweaver and Ireland 2019 rather than against intuition.
- **The albedo memory timescale.** Five years is a choice, not a
  measurement. Test how much the seasonal results move if it is one year
  or twenty.

Add a case to `test_cases.json` for each of these as you settle them.
A case with `"source": "published"` and a `reference` string is worth
several consistency checks.